# JJP LTX-2.5 — native dual-T4 test studio

Based on [AIQUEST's native LTX-2.5 Kaggle engine](https://github.com/TeamAIQ/Colab-notebooks/blob/e297aa0fb6eb1c9eaa57c855fc1cf66657b4b66b/kaggle_notebooks/ltx-2-5-22b-distilled-video-generator.ipynb), with AIQUEST credit and Gradio interface retained. Separate from your working H3, image and upscale notebooks.

1. Run **JJP_LTX25_Models.ipynb** on CPU first. Wait for the automatic private Dataset to finish processing.
2. Attach that **ONE Dataset** here. Choose **GPU T4 x2**, **Internet ON**, and stop any older studio using those GPUs.
3. Run this **one startup cell**. It verifies/stages saved models, installs the native dependencies and starts the resident engine. Open the printed Cloudflare or Gradio link. No clip runs automatically.
4. To test an image: expand **Image → Video**, upload a **Start frame**, write the scene/action/camera prompt, and press **Generate Video**. An End frame is optional for first/last-frame control. With no images it is text-to-video.
5. Start at the native **480p preset / 5 s / 8 steps**. The exact dimensions are logged: landscape text-to-video is **832×512**; image inputs use their own rounded aspect ratio. “480p” is a preset label, not an exact output height. 121 frames at 24 fps is about 5.04 s.

**Recommendation:** use one approved first image, one clear action and one camera movement for the first test. Use separate clips for edit points requiring exact character count. This edition generates its own audio; it does not synchronize to your existing song. Keep generated ambience if useful, or use your song in the editor. Audio-driven avatars and MSR are separate features, not implied here.

The INT8 transformer is tensor-parallel across both T4s; Gemma runs through the native offloader. FP32 encoder/norm protections and native INT8 kernels are retained. This test edition keeps the official eight-step schedule and refuses non-finite conditioning, latents or decoded media instead of zeroing invalid reference values. Kernel availability and timings are reported by the native engine; no speedup or model-quality guarantee is made.

The **36.867 GiB** model set is staged from the attached Dataset to verified local scratch; no inference-time model downloads. Staging requires the missing files plus a 6 GiB reserve on one writable filesystem. Temporary space is measured; do not assume a fixed Kaggle quota. Outputs are MP4 + generated AAC audio under `/kaggle/working/jjp-ltx25/outputs/`; preserve your outputs before ending the session. The model Dataset does not save generated videos.

Initial engine load occurs once; leave the cell running between clips. Native Stop cancels a generation. Editable limits are 20 minutes for engine startup and 30 minutes per clip; they stop only this notebook's owned processes, preserve completed clips, and are **spending limits, not runtime promises**. Setup/staging time is separate.

No full-size T4 inference, visual quality, live Dataset upload or public tunnel was executed in development. Local validation covers source identity, generated notebooks, storage, actual SDK upload control flow with remote calls stubbed, and numerical-guard behavior.


In [ ]:
MAX_STARTUP_SECONDS = 1200
MAX_JOB_SECONDS = 1800
from pathlib import Path
import base64,io,zipfile,sys,importlib
WORK=Path('/kaggle/working/jjp-ltx25')
RUNTIME=WORK/'runtime'
RUNTIME.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as archive:
    for name in archive.namelist():
        if Path(name).name!=name:
            raise RuntimeError('Invalid embedded runtime path')
        (RUNTIME/name).write_bytes(archive.read(name))
sys.path.insert(0,str(RUNTIME))
importlib.invalidate_caches()
for name in ('ltx25_runtime','ltx25_common','ltx25_dataset','ltx25_check_dataset'):
    previous=sys.modules.get(name)
    if previous is not None and Path(getattr(previous,'__file__','')).parent==RUNTIME:
        sys.modules.pop(name,None)

from ltx25_runtime import run
run(RUNTIME,WORK,MAX_STARTUP_SECONDS,MAX_JOB_SECONDS)
